# Fase 4 — Validación de los datos sintéticos de Golazo

**Objetivo:** comprobar que los datos sintéticos generados son **coherentes**: que respetan las cifras reales conocidas del cliente (301.000 vistas y 900 suscriptores en 30 días), que ninguna métrica sale negativa o fuera de rango, y que las distribuciones (demografía, tráfico, retención) tienen sentido antes de pasar al diseño de la base de datos.

**Requisito previo:**
```
python -m src.generador_sintetico
```
Esto genera en `/synthetic`:
- `golazo_catalogo_videos.csv`
- `golazo_analytics_sintetico.json`

In [ ]:
import json
import os

import pandas as pd

SYNTHETIC_DIR = os.path.join("..", "synthetic")

df_catalogo = pd.read_csv(os.path.join(SYNTHETIC_DIR, "golazo_catalogo_videos.csv"))

with open(os.path.join(SYNTHETIC_DIR, "golazo_analytics_sintetico.json"), encoding="utf-8") as f:
    analytics = json.load(f)

print(f"Vídeos en el catálogo: {len(df_catalogo)} (real: 338)")

## 1. Coherencia con las cifras reales conocidas

In [ ]:
views_generadas = sum(row[1] for row in analytics["evolucion_diaria"]["rows"])
subs_generados = sum(row[4] for row in analytics["evolucion_diaria"]["rows"])

print(f"Vistas generadas (30 días): {views_generadas:,} | Real: 301,000 | Coincide: {views_generadas == 301000}")
print(f"Suscriptores generados (30 días): {subs_generados} | Real: 900 | Coincide: {subs_generados == 900}")
print(f"Rango de fechas del catálogo: {df_catalogo.fecha_publicacion.min()} -> {df_catalogo.fecha_publicacion.max()}")

## 2. Ningún valor negativo ni fuera de rango

Comprobación defensiva: ninguna métrica de conteo (vistas, suscriptores, likes) debe ser negativa, y los porcentajes deben sumar 100.

In [ ]:
# Catálogo de vídeos
assert (df_catalogo[["views_totales", "likes", "comentarios", "duracion_segundos"]] >= 0).all().all()

# Evolución diaria
assert all(row[1] >= 0 and row[4] >= 0 and row[5] >= 0 for row in analytics["evolucion_diaria"]["rows"])

# Tráfico: no negativos y suma exacta
views_trafico = [row[1] for row in analytics["trafico"]["rows"]]
assert all(v >= 0 for v in views_trafico)
assert sum(views_trafico) == 301000

# Demografía: suma ~100
suma_demografia = sum(row[2] for row in analytics["demografia"]["rows"])
assert 99.5 <= suma_demografia <= 100.5

print("Todas las comprobaciones de rango pasaron correctamente.")

## 3. Vistazo a las distribuciones

In [ ]:
print("Vídeos por categoría de contenido:")
print(df_catalogo["categoria"].value_counts())
print("\nVistas medias por categoría:")
print(df_catalogo.groupby("categoria")["views_totales"].mean().round(0).sort_values(ascending=False))

In [ ]:
df_trafico = pd.DataFrame(
    analytics["trafico"]["rows"],
    columns=[c["name"] for c in analytics["trafico"]["columnHeaders"]],
)
df_trafico["pct"] = (df_trafico["views"] / df_trafico["views"].sum() * 100).round(1)
df_trafico.sort_values("views", ascending=False)

In [ ]:
df_retencion = pd.DataFrame(
    analytics["retencion_audiencia"]["rows"],
    columns=[c["name"] for c in analytics["retencion_audiencia"]["columnHeaders"]],
)

# Curva de retención media entre todos los vídeos de la muestra
curva_media = df_retencion.groupby("elapsedVideoTimeRatio")["audienceWatchRatio"].mean()
curva_media

## Conclusión

Los datos sintéticos respetan las cifras reales conocidas del cliente, no contienen valores negativos ni distribuciones que sumen fuera de rango, y siguen patrones de forma (duración, ratios de engagement) heredados de datos reales de @PuroBalompie. Con el catálogo de vídeos y los 5 informes de Analytics ya generados y validados, el siguiente paso es diseñar el modelo entidad-relación de la base de datos (Fase 5) a partir de las entidades ya identificadas: canal, vídeo, evolución diaria, retención, demografía, tráfico e ingresos.